In [1]:
import GPUtil
from pynvml import *
import torch

In [2]:
device_id = torch.device("cuda:1")

In [3]:
nvmlInit()
h = nvmlDeviceGetHandleByIndex(0)
info = nvmlDeviceGetMemoryInfo(h)
total_nvml = int(info.total / (1024 * 1024))
used_nvml = int(info.used / (1024 * 1024))
cuda_context_mem = used_nvml - GPUtil.getGPUs()[0].memoryUsed
framework_initial_mem = GPUtil.getGPUs()[0].memoryUsed

In [4]:
import transformers
model = transformers.AutoModelForCausalLM.from_pretrained("facebook/opt-125m").to("cuda")


/home/glaswegian/miniconda3/envs/llmem/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/glaswegian/miniconda3/envs/llmem/lib/python3.9/site-packages/huggingface_hub/file_download.py:896: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/home/glaswegian/miniconda3/envs/llmem/lib/python3.9/site-packages/huggingface_hub/file_download.py:896: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/home/glaswegian/miniconda3/envs/llmem/lib/python3.9/site-packages/torch/cuda/__init__.py:235: UserWarning: 

In [5]:
torch.cuda.empty_cache()

In [6]:
model.gradient_checkpointing_enable()

In [7]:
tokenizer = transformers.AutoTokenizer.from_pretrained(
        "facebook/opt-125m",
        model_max_length=128,
        padding_side="right",
        use_fast=False,
    )
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
elif tokenizer.eos_token is None:  # for bert
    tokenizer.eos_token = tokenizer.pad_token

In [8]:
from datasets import load_dataset
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, DataCollatorForLanguageModeling
def prepare_data() -> DataLoader:
    model_name = "facebook/opt-125m"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.pad_token = tokenizer.eos_token
    dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="train")

    # 3. participle
    def tokenize_function(examples):
        return tokenizer(examples["text"], truncation=True, max_length=128)

    tokenized_datasets = dataset.map(tokenize_function, batched=True, remove_columns=["text"])

    # 4. create DataCollatorForLanguageModeling
    data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

    # 5. create DataLoader
    tokenized_datasets.set_format("torch")
    dataloader = DataLoader(tokenized_datasets, batch_size=10, shuffle=True, collate_fn=data_collator)
    return dataloader

In [9]:
dl = prepare_data()
booster_kwargs = {}

config = {
    "batch_size": 10,
    "lr": 1e-5,
    "epochs": 1,
    "warmup_ratio": 0.,
    "weight_decay": 0.03,
}

def move_to_cuda(batch, device):
    return {k: v.to(device) for k, v in batch.items()}

#
for batch in dl:
    if batch["input_ids"].size()[1] == 128:
        test_long_input = move_to_cuda(batch, torch.cuda.current_device())
        break

/home/glaswegian/miniconda3/envs/llmem/lib/python3.9/site-packages/huggingface_hub/file_download.py:896: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [10]:
# Set lr scheduler
total_steps = len(dl) * config["epochs"]
num_warmup_steps = int(config["warmup_ratio"] * total_steps)

In [11]:
from transformers import get_cosine_schedule_with_warmup
optimizer = torch.optim.AdamW(model.parameters(), lr=config["lr"], weight_decay=0.0)
# Set lr scheduler
lr_scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=len(dl) * config["epochs"],
)

In [12]:
from experiments.baselines.LLmem_startup import SizeEstimator
real_bs = 0 # For batch size search mode
# real_bs = test_long_input["input_ids"].size()[0] # To estimate with specific batch size
se = SizeEstimator(model, test_long_input["input_ids"][0:2], real_bs, bytes=2, bytes_input=8,
                   gpu_n=1, tp=0, lm_fp32=True, m_total=total_nvml)
# gpu_n: total number of GPUs
torch.cuda.empty_cache()
prev_get_output = GPUtil.getGPUs()[0].memoryUsed
se.get_output_sizes()
torch.cuda.empty_cache()
after_get_output = GPUtil.getGPUs()[0].memoryUsed

RuntimeError: CUDA error: no kernel image is available for execution on the device
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [13]:
from colossalai.booster import Booster
booster = Booster(**booster_kwargs)
model, optimizer, _, _, _ = booster.boost(model, optimizer)
torch.cuda.empty_cache()

/home/glaswegian/miniconda3/envs/llmem/lib/python3.9/site-packages/colossalai/utils/safetensors.py:13: UserWarning: Please install the latest tensornvme to use async save. pip install git+https://github.com/hpcaitech/TensorNVMe.git
  warnings.warn(
/home/glaswegian/miniconda3/envs/llmem/lib/python3.9/site-packages/colossalai/shardformer/layer/normalization.py:48: UserWarning: Please install apex from source (https://github.com/NVIDIA/apex) to use the fused RMSNorm kernel
  warnings.warn("Please install apex from source (https://github.com/NVIDIA/apex) to use the fused RMSNorm kernel")
/home/glaswegian/miniconda3/envs/llmem/lib/python3.9/site-packages/colossalai/shardformer/layer/normalization.py:93: UserWarning: Please install apex from source (https://github.com/NVIDIA/apex) to use the fused RMSNorm kernel
  warnings.warn(
